# VF pipeline v2 — fixes for the first-draft problems

One notebook that **re-imports the model + data** and rebuilds the pipeline so that the paper's numbers come from a clean, leakage-aware run.

| Problem # | What it fixes | Cell / section |
|---|---|---|
| **1** | Gene-ID / coordinate bug (Prodigal coords now read from the FASTA headers, no positional remap) | §8 genome load (asserts) |
| **2** | Domain-feature mismatch (same CDD library + same parser + same threshold for train and genome; E-value-size dependence removed) | §6, §8 |
| **3** | Homology-aware split (MMseqs2 clusters, train/val/test = 80/10/10, no cluster shared) + leakage quantification of the old random split | §4 |
| **5** | BLAST leakage: self-hit removal diagnostics, identity-cap and family-level de-leaking variants, train-vs-genome feature shift | §6, §11 |
| **6** | Negative-set audit + shortcut baselines (length, composition) | §5 |
| **7** | Calibration, precision/recall at thresholds, prevalence-adjusted expected false positives | §12 |
| **11** | Baselines: length/composition, BLAST-only, CDD-only, BLAST+CDD, frozen ESM-2 + LR, head, embedding-only, M2, M3 | §7, §10, §11 |
| **12** | Seeds (mean ± SD), cluster-aware paired bootstrap CIs | §10, §11 |
| **14** | All genome analyses re-run on the Prodigal gene set with true coordinates | §13 |

**Run order:** set `CFG["smoke"] = True` and run the whole notebook once (≈5–10 min on a T4). If it finishes, set it to `False` and run for real (≈35–50 min per seed on a T4; each epoch is checkpointed to Drive and the run resumes automatically).

**Inputs it looks for** (in `AI/public projects/`, `AI/public_projects/`, and their `checkpoints/` folders): `VF_positive_subset_final_2.faa`, `VF_negative_subset_final.faa`, `train_vs_vfdb.tsv`, `train_cdd_hits.tblout` (raw hmmsearch `--domtblout`), `predicted_orfs_new.faa`, `new_orfs_vs_vfdb.tsv`, `new_cdd_hits.tblout`.

In [ ]:
# §1  Installs (Colab already has most of these)
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("biopython", "xgboost", "scikit-learn", "scipy", "transformers", "upsetplot")

In [ ]:
# §2  Config, imports, paths
import os, re, json, math, time, random, shutil, subprocess, pickle, warnings, sys
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, f1_score,
                             brier_score_loss, roc_curve, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from scipy.stats import ks_2samp, spearmanr
import xgboost as xgb
warnings.filterwarnings("ignore")

CFG = dict(
    smoke=False,                      # True = tiny end-to-end test. RUN THIS FIRST.
    seeds=[42, 43, 44],               # item 12: >=3 seeds for mean +- SD
    model_name="facebook/esm2_t12_35M_UR50D",
    epochs=6, batch_size=16, lr=2e-5, beta=0.3, tau=0.1, max_len=512, unfreeze=2,
    use_amp=True,                     # fp16 autocast on GPU (speed); set False to match the old fp32 run
    split_identity=0.30, split_cov=0.8,   # MMseqs2 identity/coverage that defines "homologous"
    leak_identities=[0.9, 0.5, 0.3],      # identities used to quantify leakage of the OLD random split
    n_folds=10,                           # 10 folds -> 8 train / 1 val / 1 test
    split_seed=42,
    retrain_random=False,             # True = also retrain on a random 80/10/10 split for a like-for-like comparison
    dom_score_min=25.0,               # CDD feature: count domain hits with DOMAIN bit-score >= this (database-size independent)
    primary_model="m3_self",          # model used for calibration + genome tiers: m3_self | m3_cap90 | m3_family | m2 | emb_only
    hc_threshold=0.8,                 # "high-confidence" cut-off on the raw score (as in the draft)
    prevalences=[0.01, 0.02, 0.05, 0.10],
    frozen_baseline=True,             # frozen ESM-2 mean-pool + logistic regression
    boot=1000,
    locus_gap_bp=500,
    threads=4,
    smoke_n_per_class=600, smoke_n_genes=300,
)
if CFG["smoke"]:
    CFG.update(seeds=[42], epochs=1, boot=100, leak_identities=[0.5, 0.3], frozen_baseline=False, n_folds=5)

# ---- where things live ------------------------------------------------------------------
IN_COLAB = os.path.exists("/content")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass
WORK = Path("/content") if IN_COLAB else Path("/tmp")

EXTRA_DIRS = []   # <- add your own folders here if the auto-search below does not find your files
_cands = []
if os.environ.get("VF_DATA_DIR"):
    _cands.append(Path(os.environ["VF_DATA_DIR"]))
for b in ["/content/drive/MyDrive/AI/public projects/",
          "/content/drive/MyDrive/AI/public_projects/",
          "/content/drive/MyDrive/AI/public_projects/VF_WORK/"]:
    _cands += [Path(b), Path(b) / "checkpoints"]
SEARCH_DIRS = [Path(d) for d in EXTRA_DIRS] + [d for d in _cands if d.exists()]

def find_file(name, required=True):
    for d in SEARCH_DIRS:
        p = d / name
        if p.exists():
            return p
    if required:
        raise FileNotFoundError(f"{name} not found in: {[str(d) for d in SEARCH_DIRS]}")
    return None

ROOT = SEARCH_DIRS[0] if SEARCH_DIRS else WORK / "work"
OUT = ROOT / "rerun_v2" / ("smoke" if CFG["smoke"] else "full")
for sub in ["cache", "ckpt", "figures", "tables"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print("search dirs :", [str(d) for d in SEARCH_DIRS])
print("outputs  ->", OUT)
print("config      :", json.dumps({k: v for k, v in CFG.items()}, default=str))

In [ ]:
# §3  Utilities + load the labelled data (order is preserved: train_0 ... train_N-1 = positives then negatives,
#     exactly the IDs used in train_vs_vfdb.tsv and train_cdd_hits.tblout)
def save_table(d, name):
    d.to_csv(OUT / "tables" / f"{name}.csv", index=True)
    return d

def eval_probs(y, p, thr=0.5):
    return dict(auc=roc_auc_score(y, p), ap=average_precision_score(y, p),
                acc=accuracy_score(y, p >= thr), f1=f1_score(y, p >= thr), brier=brier_score_loss(y, p))

def score_metrics(y, s):          # for raw scores that are not probabilities (bitscore, domain count)
    return dict(auc=roc_auc_score(y, s), ap=average_precision_score(y, s))

def fmt_ms(vals):
    vals = np.asarray(vals, float)
    return f"{vals.mean():.4f} ± {vals.std(ddof=1) if len(vals) > 1 else 0.0:.4f}"

def ece_score(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e

def paired_bootstrap_delta(y, pa, pb, n=1000, seed=0, groups=None):
    """AUC(a) - AUC(b) with a percentile CI. If `groups` is given, whole clusters are resampled
    (sequences in one cluster are not independent)."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y); N = len(y)
    if groups is not None:
        groups = np.asarray(groups)
        members = {g: np.where(groups == g)[0] for g in np.unique(groups)}
        keys = np.array(list(members.keys()))
    d = []
    for _ in range(n):
        if groups is None:
            idx = rng.integers(0, N, N)
        else:
            pick = rng.choice(keys, size=len(keys), replace=True)
            idx = np.concatenate([members[g] for g in pick])
        if len(np.unique(y[idx])) < 2:
            continue
        d.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.asarray(d)
    return dict(delta=roc_auc_score(y, pa) - roc_auc_score(y, pb), lo=np.percentile(d, 2.5),
                hi=np.percentile(d, 97.5), p_le0=float((d <= 0).mean()))

def boot_auc_ci(y, p, n=1000, seed=0, groups=None):
    rng = np.random.default_rng(seed); y = np.asarray(y); N = len(y); a = []
    if groups is not None:
        groups = np.asarray(groups); members = {g: np.where(groups == g)[0] for g in np.unique(groups)}
        keys = np.array(list(members.keys()))
    for _ in range(n):
        idx = rng.integers(0, N, N) if groups is None else np.concatenate(
            [members[g] for g in rng.choice(keys, size=len(keys), replace=True)])
        if len(np.unique(y[idx])) > 1:
            a.append(roc_auc_score(y[idx], p[idx]))
    return roc_auc_score(y, p), np.percentile(a, 2.5), np.percentile(a, 97.5)

def load_labelled():
    rows = []
    for label, fname in [(1, "VF_positive_subset_final_2.faa"), (0, "VF_negative_subset_final.faa")]:
        for rec in SeqIO.parse(str(find_file(fname)), "fasta"):
            rows.append((label, rec.id, rec.description, str(rec.seq)))
    d = pd.DataFrame(rows, columns=["label", "orig_id", "desc", "seq"])
    d.insert(0, "tid", [f"train_{i}" for i in range(len(d))])      # <- must match the BLAST/HMM query IDs
    d["vfg"] = d.orig_id.str.extract(r"^(VFG\d+)")[0]
    d["len"] = d.seq.str.len()
    return d

df_full = load_labelled()
print(f"loaded {len(df_full)} sequences | positives {int(df_full.label.sum())} | negatives {int((1-df_full.label).sum())}")
if CFG["smoke"]:
    df = (df_full.groupby("label", group_keys=False).sample(n=min(CFG["smoke_n_per_class"], int(df_full.label.value_counts().min())),
                                                           random_state=0).reset_index(drop=True))
    print("SMOKE MODE: using", len(df), "sequences")
else:
    df = df_full.reset_index(drop=True)
y_all = df.label.values.astype(int)

In [ ]:
# §4  (item 3)  Homology-aware split with MMseqs2 + quantify the leakage of the OLD random split
def mmseqs_bin():
    exe = shutil.which("mmseqs")
    if exe:
        return exe
    subprocess.run(["apt-get", "install", "-y", "-qq", "mmseqs2"], capture_output=True, text=True)
    exe = shutil.which("mmseqs")
    if exe:
        return exe
    tgz = str(WORK / "mmseqs.tar.gz")
    for flavour in ["avx2", "sse41"]:
        subprocess.run(["wget", "-q", f"https://mmseqs.com/latest/mmseqs-linux-{flavour}.tar.gz", "-O", tgz], capture_output=True)
        subprocess.run(["tar", "xzf", tgz, "-C", str(WORK)], capture_output=True)
        cand = str(WORK / "mmseqs" / "bin" / "mmseqs")
        if os.path.exists(cand) and subprocess.run([cand, "version"], capture_output=True).returncode == 0:
            os.environ["PATH"] += ":" + str(WORK / "mmseqs" / "bin")
            return cand
    raise RuntimeError("could not install mmseqs2 - install it manually and re-run")

def cluster_sequences(d, identity, cov=0.8):
    sens = 7.5 if identity <= 0.35 else (5.7 if identity <= 0.6 else 4.0)
    tag = f"n{len(d)}_id{int(round(identity*100))}_c{int(round(cov*100))}"
    cache = OUT / "cache" / f"clusters_{tag}.tsv"
    if not cache.exists():
        work = WORK / "mmseqs_work" / ("smoke_" if CFG["smoke"] else "") / tag
        shutil.rmtree(work, ignore_errors=True); work.mkdir(parents=True, exist_ok=True)
        with open(work / "seqs.faa", "w") as f:
            for tid, s in zip(d.tid, d.seq):
                f.write(f">{tid}\n{s}\n")
        cmd = [mmseqs_bin(), "easy-cluster", str(work / "seqs.faa"), str(work / "out"), str(work / "tmp"),
               "--min-seq-id", str(identity), "-c", str(cov), "--cov-mode", "0", "-s", str(sens),
               "--threads", str(CFG["threads"])]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-1500:])
        shutil.copy(work / "out_cluster.tsv", cache)
    m = pd.read_csv(cache, sep="\t", header=None, names=["rep", "member"])
    rep_of = dict(zip(m.member, m.rep))
    cl = d.tid.map(rep_of)
    assert cl.notna().all(), "some sequences are missing from the clustering output"
    return cl.factorize()[0]

def group_split(labels, groups, n_folds, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold = np.full(len(labels), -1)
    for k, (_, te) in enumerate(sgkf.split(np.zeros(len(labels)), labels, groups)):
        fold[te] = k
    split = np.array(["train"] * len(labels), dtype=object)
    split[fold == 0] = "test"; split[fold == 1] = "val"
    return split

def random_split3(labels, seed):
    idx = np.arange(len(labels))
    tr, rest = train_test_split(idx, test_size=0.2, random_state=seed, stratify=labels)
    va, te = train_test_split(rest, test_size=0.5, random_state=seed, stratify=labels[rest])
    s = np.array(["train"] * len(labels), dtype=object); s[va] = "val"; s[te] = "test"
    return s

# clusters at several identities (for leakage measurement) + the one used for the split
CLUSTERS = {}
for ident in sorted(set(CFG["leak_identities"] + [CFG["split_identity"]]), reverse=True):
    CLUSTERS[ident] = cluster_sequences(df, ident, CFG["split_cov"])
    print(f"identity {ident:.2f}: {len(np.unique(CLUSTERS[ident]))} clusters for {len(df)} sequences")
groups = CLUSTERS[CFG["split_identity"]]

cluster_split = group_split(y_all, groups, CFG["n_folds"], CFG["split_seed"])
# the OLD recipe from the first draft: stratified random 80/20, seed 42 (train vs "test")
_idx = np.arange(len(df))
_tr, _te = train_test_split(_idx, test_size=0.2, random_state=42, stratify=y_all)
old_random_split = np.array(["train"] * len(df), dtype=object); old_random_split[_te] = "test"
random3 = random_split3(y_all, CFG["split_seed"])

def leak_frac(split, eval_name, ident):
    g = CLUSTERS[ident]
    tr_groups = np.unique(g[split == "train"])
    ev = split == eval_name
    return float(np.isin(g[ev], tr_groups).mean())

rows = []
for name, sp in [("old random 80/20", old_random_split), ("random 80/10/10", random3), ("cluster (new)", cluster_split)]:
    r = {"split": name, "n_train": int((sp == "train").sum()), "n_test": int((sp == "test").sum()),
         "test_pos_frac": float(y_all[sp == "test"].mean())}
    for ident in sorted(CLUSTERS, reverse=True):
        r[f"test_seqs_with_train_cluster_mate@{int(round(ident*100))}%"] = leak_frac(sp, "test", ident)
    rows.append(r)
leak_tbl = save_table(pd.DataFrame(rows).set_index("split"), "split_leakage")
print(leak_tbl.T.to_string())
assert leak_frac(cluster_split, "test", CFG["split_identity"]) == 0.0 and leak_frac(cluster_split, "val", CFG["split_identity"]) == 0.0
print("\nOK: no train sequence shares a cluster with val/test at identity", CFG["split_identity"])
print("split sizes:", {s: int((cluster_split == s).sum()) for s in ["train", "val", "test"]},
      "| positive fraction:", {s: round(float(y_all[cluster_split == s].mean()), 3) for s in ["train", "val", "test"]})

In [ ]:
# §5  (item 6)  Negative-set audit + shortcut baselines
HK_KEYWORDS = {   # same families as the draft; edit/extend and quote this list in the paper
    "elongation factor": ["elongation factor"],
    "ribosomal protein": ["ribosomal protein", "ribosomal rna"],
    "chaperonin/heat shock": ["chaperonin", "groel", "groes", "heat shock", "dnak", "hsp"],
    "ATP synthase": ["atp synthase"],
    "RNA polymerase (core)": ["dna-directed rna polymerase", "rna polymerase subunit"],
    "translation initiation/release factor": ["initiation factor", "release factor"],
    "DNA gyrase/topoisomerase": ["dna gyrase", "topoisomerase"],
    "ABC transporter (general)": ["abc transporter", "atp-binding cassette"],
}
def classify_family(text):
    t = str(text).lower()
    for fam, kws in HK_KEYWORDS.items():
        if any(k in t for k in kws):
            return fam
    return None

AA = "ACDEFGHIKLMNPQRSTVWY"
def comp_features(seqs):
    X = np.zeros((len(seqs), 21), dtype=np.float32)
    for i, s in enumerate(seqs):
        c = Counter(s); L = max(len(s), 1)
        X[i, :20] = [c.get(a, 0) / L for a in AA]; X[i, 20] = len(s)
    return X

audit = {}
pos, neg = df[df.label == 1], df[df.label == 0]
audit["n_pos"], audit["n_neg"] = len(pos), len(neg)
audit["len_mean_pos/neg"] = (round(float(pos.len.mean()), 1), round(float(neg.len.mean()), 1))
audit["len_median_pos/neg"] = (float(pos.len.median()), float(neg.len.median()))
audit["len_KS(stat,p)"] = tuple(round(float(x), 4) for x in ks_2samp(pos.len, neg.len))
audit["frac_with_VFG_id pos/neg"] = (round(float(pos.vfg.notna().mean()), 3), round(float(neg.vfg.notna().mean()), 3))
mixed = pd.DataFrame({"g": groups, "y": y_all}).groupby("g").y.agg(["mean", "size"])
mixed_cl = mixed[(mixed["mean"] > 0) & (mixed["mean"] < 1)]
audit[f"clusters_with_both_labels@{int(CFG['split_identity']*100)}%"] = int(len(mixed_cl))
audit["seqs_in_mixed_clusters"] = int(mixed_cl["size"].sum())
txt = df.desc.str.replace(r"^\S+\s*", "", regex=True).str.lower()
audit["frac_with_description pos/neg"] = (round(float((txt[df.label == 1].str.len() > 0).mean()), 3), round(float((txt[df.label == 0].str.len() > 0).mean()), 3))
fam = txt.map(classify_family)
fam_tbl = pd.crosstab(fam.fillna("(none)"), df.label).rename(columns={0: "negatives", 1: "positives"})
print("=== dataset audit ===")
for k, v in audit.items():
    print(f"  {k}: {v}")
print("\nhousekeeping-keyword hits in the FASTA descriptions (if the files carry annotations):")
print(fam_tbl.to_string())
rng = np.random.default_rng(0)
for lab, nm in [(0, "NEGATIVE"), (1, "POSITIVE")]:
    sub = df[df.label == lab]
    print(f"\n--- 8 random {nm} headers (read them: what ARE the negatives?) ---")
    for s in sub.desc.sample(min(8, len(sub)), random_state=1):
        print("  ", s[:150])
save_table(pd.Series(audit, name="value").astype(str).to_frame(), "dataset_audit")
save_table(fam_tbl, "dataset_housekeeping_keywords")

# shortcut baselines: can length / amino-acid composition alone separate the classes?
Xc = comp_features(df.seq.tolist())
def shortcut_auc(X, split):
    tr, te = split == "train", split == "test"
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, eval_metric="logloss", random_state=0, n_jobs=-1)
    clf.fit(X[tr], y_all[tr]); return roc_auc_score(y_all[te], clf.predict_proba(X[te])[:, 1])
sc = {}
for nm, X in [("length only", Xc[:, 20:21]), ("composition + length", Xc)]:
    for sp_name, sp in [("old random split", old_random_split), ("cluster split", cluster_split)]:
        sc[(nm, sp_name)] = shortcut_auc(X, sp)
sc_tbl = save_table(pd.Series(sc).unstack().round(4), "shortcut_baselines")
print("\n=== shortcut baselines (test AUC) - if these are far above 0.5 the dataset has an artefact ===")
print(sc_tbl.to_string())

In [ ]:
# §6  (items 2, 5)  Training-side evidence features, defined identically to the genome side
def _is_float(x):
    try:
        float(x); return True
    except ValueError:
        return False

def parse_domtblout(path, mode="domscore", dom_score_min=25.0, evalue_max=1e-5):
    """hmmsearch --domtblout, sequence = column 0. mode 'domscore' counts domain hits with DOMAIN bit-score >= dom_score_min
    (does not depend on database size); mode 'evalue' reproduces the old rule (full-sequence E-value <= 1e-5 per row)."""
    counts = Counter()
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            p = line.split(None, 22)
            if len(p) < 22:
                continue
            try:
                ok = (float(p[13]) >= dom_score_min) if mode == "domscore" else (float(p[6]) <= evalue_max)
            except ValueError:
                continue
            if ok:
                counts[p[0]] += 1
    return counts

BLAST12 = ["query_id", "subject_id", "pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]
def read_blast(path):
    with open(path, errors="replace") as f:
        first = f.readline().rstrip("\n").split("\t")
    if len(first) == 13 and not _is_float(first[2]):          # outfmt "6 qseqid sseqid stitle pident ..."
        cols = ["query_id", "subject_id", "subject_title"] + BLAST12[2:]
    else:
        cols = BLAST12 + [f"extra{i}" for i in range(max(0, len(first) - 12))]
    h = pd.read_csv(path, sep="\t", names=cols, header=None, quoting=3, encoding_errors="replace", on_bad_lines="skip", low_memory=False)
    h["subject_vfg"] = h.subject_id.astype(str).str.extract(r"^(VFG\d+)")[0]
    return h

def bitscore_feature(hits, ids, own_vfg=None, id_cap=None, tid2cl=None, vfg2cl=None):
    """Best VFDB bitscore per query (0 if none). De-leaking options for TRAINING proteins:
       own_vfg : drop hits to the protein's own VFDB record (matched by VFG number)
       id_cap  : additionally drop hits with identity >= id_cap
       tid2cl/vfg2cl : additionally drop hits to VFDB records that sit in the same homology cluster as the query ("family" variant)"""
    h = hits
    if own_vfg is not None:
        qv = h.query_id.map(own_vfg)
        h = h[~(qv.notna() & (qv == h.subject_vfg))]
    if id_cap is not None:
        h = h[h.pident < id_cap]
    if tid2cl is not None:
        qc, sc_ = h.query_id.map(tid2cl), h.subject_vfg.map(vfg2cl)
        h = h[~(qc.notna() & sc_.notna() & (qc == sc_))]
    best = h.groupby("query_id").bitscore.max()
    return pd.Series(list(ids)).map(best).fillna(0.0).values

# ---- CDD domain count (training) -----------------------------------------------------
cdd_path = find_file("train_cdd_hits.tblout", required=False) or find_file("train_cdd_hits.tlout", required=False)
if cdd_path is None:
    raise FileNotFoundError("train_cdd_hits.tblout (raw hmmsearch --domtblout of the training sequences) not found. Re-run: "
                            "hmmsearch --cpu 2 --noali --domtblout train_cdd_hits.tblout virulence_factors_CDD_repaired.hmm train_sequences.faa")
cnt_dom = parse_domtblout(cdd_path, "domscore", CFG["dom_score_min"])
cnt_old = parse_domtblout(cdd_path, "evalue")
cdd_train = df.tid.map(cnt_dom).fillna(0).values.astype(float)
cdd_train_old = df.tid.map(cnt_old).fillna(0).values.astype(float)
print(f"CDD file: {cdd_path.name}")
print(f"genes with >=1 domain  (new rule, dom score>={CFG['dom_score_min']}): pos {np.mean(cdd_train[y_all==1]>0):.3f} | neg {np.mean(cdd_train[y_all==0]>0):.3f}")
print(f"genes with >=1 domain  (old rule, seq E<=1e-5)               : pos {np.mean(cdd_train_old[y_all==1]>0):.3f} | neg {np.mean(cdd_train_old[y_all==0]>0):.3f}")

# ---- BLAST bitscore (training) ---------------------------------------------------------
tb_path = find_file("train_vs_vfdb.tsv")
tb = read_blast(tb_path)
tb = tb[tb.query_id.isin(set(df.tid))].copy()
own = dict(zip(df.tid[df.vfg.notna()], df.vfg[df.vfg.notna()]))
tid2cl = dict(zip(df.tid, groups))
vfg2cl = dict(zip(df.vfg[df.vfg.notna()], groups[df.vfg.notna().values]))
tb["own_vfg"] = tb.query_id.map(own)
is_self = tb.own_vfg.notna() & (tb.own_vfg == tb.subject_vfg)
n_with_vfg = int(df.vfg.notna().sum())
print(f"\nBLAST file: {tb_path.name}: {len(tb)} hits | self hits removed: {int(is_self.sum())} "
      f"| sequences with a VFG id: {n_with_vfg} | sequences that actually self-hit: {tb[is_self].query_id.nunique()}")
rest = tb[~is_self]
near = rest[rest.pident >= 99.5].merge(df[["tid", "label"]], left_on="query_id", right_on="tid")
print(f"non-self hits at >=99.5% identity left after self removal: {len(near)} (queries: {near.query_id.nunique()}; "
      f"positives {int((near.drop_duplicates('query_id').label==1).sum())}, negatives {int((near.drop_duplicates('query_id').label==0).sum())})")

BITS = {  # training features, one per de-leaking variant
    "self":   bitscore_feature(tb, df.tid, own_vfg=own),
    "cap90":  bitscore_feature(tb, df.tid, own_vfg=own, id_cap=90.0),
    "family": bitscore_feature(tb, df.tid, own_vfg=own, tid2cl=tid2cl, vfg2cl=vfg2cl),
}
rows = []
for v, b in BITS.items():
    rows.append({"variant": v, "frac_nonzero_pos": float((b[y_all == 1] > 0).mean()), "frac_nonzero_neg": float((b[y_all == 0] > 0).mean()),
                 "mean_bits_pos": float(b[y_all == 1].mean()), "mean_bits_neg": float(b[y_all == 0].mean()),
                 "AUC_alone_(all)": roc_auc_score(y_all, b)})
print("\n=== BLAST feature under each de-leaking rule (training set) ===")
print(save_table(pd.DataFrame(rows).set_index("variant").round(3), "blast_feature_variants").to_string())

In [ ]:
# §7  (item 11)  Classical baselines on the SAME cluster split (no GPU needed)
tr_m, va_m, te_m = [cluster_split == s for s in ("train", "val", "test")]
BASELINES = {}
def _xgb(seed=0, **kw):
    return xgb.XGBClassifier(n_estimators=200, max_depth=kw.get("max_depth", 6), learning_rate=0.05,
                             eval_metric="logloss", random_state=seed, n_jobs=-1)
yt = y_all[te_m]
for nm, X in [("length_only", Xc[:, 20:21]), ("composition+length", Xc)]:
    p = _xgb(0, max_depth=4).fit(X[tr_m], y_all[tr_m]).predict_proba(X[te_m])[:, 1]
    BASELINES[nm] = eval_probs(yt, p)
for v, b in BITS.items():
    BASELINES[f"blast_only_{v}"] = score_metrics(yt, b[te_m])
BASELINES["cdd_only"] = score_metrics(yt, cdd_train[te_m])
for v, b in BITS.items():
    X = np.c_[cdd_train, b]
    p = _xgb(0, max_depth=3).fit(X[tr_m], y_all[tr_m]).predict_proba(X[te_m])[:, 1]
    BASELINES[f"blast+cdd_{v}"] = eval_probs(yt, p)
bl_tbl = save_table(pd.DataFrame(BASELINES).T.round(4), "baselines_classical")
print("=== classical baselines (cluster-split test) ===")
print(bl_tbl.to_string())

In [ ]:
# §8  (items 1, 2)  Genome side: Prodigal genes with coordinates taken from the FASTA headers - NO positional remap
def parse_prodigal_faa(path):
    rows = []
    for rec in SeqIO.parse(str(path), "fasta"):
        parts = [x.strip() for x in rec.description.split(" # ")]
        if len(parts) < 5:
            raise ValueError(f"unexpected Prodigal header: {rec.description[:100]}")
        m = re.search(r"partial=(\d\d)", parts[4])
        rows.append(dict(gene_id=rec.id, contig=rec.id.rsplit("_", 1)[0], start=int(parts[1]), end=int(parts[2]),
                         strand=int(parts[3]), partial=m.group(1) if m else "NA", seq=str(rec.seq).rstrip("*")))
    g = pd.DataFrame(rows)
    g["aa_len"] = g.seq.str.len()
    return g

faa_path = find_file("predicted_orfs_new.faa")
genes = parse_prodigal_faa(faa_path)
assert genes.gene_id.is_unique
gid_all = set(genes.gene_id)                                                   # ID check is done against ALL genes
if CFG["smoke"]:
    genes = genes.head(CFG["smoke_n_genes"]).reset_index(drop=True)
# --- coordinate sanity check: nucleotide span must match the protein length (the old remap failed this) ---
span = genes.end - genes.start + 1
ok = (span - 3 * (genes.aa_len + 1)).abs() <= 3
print(f"{len(genes)} genes | coordinate check (span == 3*(aa+1) +-3 nt): {ok.mean():.4f}")
assert ok.mean() > 0.97, "Prodigal coordinates do not match protein lengths - check the FASTA headers"

gblast = read_blast(find_file("new_orfs_vs_vfdb.tsv"))
gcdd_path = find_file("new_cdd_hits.tblout")
g_cnt = parse_domtblout(gcdd_path, "domscore", CFG["dom_score_min"])         # SAME parser + threshold as training
g_cnt_old = parse_domtblout(gcdd_path, "evalue")
gid_set = set(genes.gene_id)
ov_b = gblast.query_id.isin(gid_all).mean() if len(gblast) else 1.0
ov_c = np.mean([k in gid_all for k in g_cnt]) if len(g_cnt) else 1.0
print(f"ID agreement with the FASTA (must be ~1.0, was 0.0 before the fix): BLAST {ov_b:.4f} | CDD {ov_c:.4f}")
assert ov_b > 0.99 and ov_c > 0.99, "BLAST/HMM IDs do not match the FASTA IDs"
gblast = gblast[gblast.query_id.isin(gid_set)].copy()

g_cdd = genes.gene_id.map(g_cnt).fillna(0).values.astype(float)
g_bits = bitscore_feature(gblast, genes.gene_id)                              # genes are not VFDB records -> no self-hit removal
print(f"genes with a VFDB hit: {gblast.query_id.nunique()} ({gblast.query_id.nunique()/len(genes):.1%}) | "
      f"genes with >=1 CDD domain: new rule {int((g_cdd>0).sum())} ({(g_cdd>0).mean():.1%}), "
      f"old E-value rule {int(genes.gene_id.map(g_cnt_old).fillna(0).gt(0).sum())}")

In [ ]:
# §9  Model definition (identical architecture to the draft, minus the unused window encoder) + training / inference helpers
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(CFG["use_amp"]) and DEVICE.type == "cuda"
print("device:", DEVICE, "| AMP:", USE_AMP)
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def free_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class AttentionPooling(nn.Module):
    def __init__(self, d):
        super().__init__(); self.attention = nn.Linear(d, 1, bias=False)
    def forward(self, x, mask):
        s = self.attention(x).squeeze(-1).float().masked_fill(mask == 0, -1e9)
        w = F.softmax(s, dim=-1)
        return torch.sum(x * w.unsqueeze(-1), dim=1), w

class GeneEncoder(nn.Module):
    def __init__(self, name, unfreeze_last_n_layers=2):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(name)
        self.hidden_dim = self.transformer.config.hidden_size
        self.n_layers = self.transformer.config.num_hidden_layers
        for p in self.transformer.parameters():
            p.requires_grad = False
        for pname, p in self.transformer.named_parameters():
            for i in range(self.n_layers - unfreeze_last_n_layers, self.n_layers):
                if f"layer.{i}." in pname:
                    p.requires_grad = True
        self.layer_weight_logits = nn.Parameter(torch.zeros(self.n_layers + 1))
        self.attn_pool = AttentionPooling(self.hidden_dim)
    def forward(self, input_ids, attention_mask):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        hs = torch.stack([h.float() for h in out.hidden_states], 0)
        w = F.softmax(self.layer_weight_logits, dim=0)
        comb = (hs * w.view(-1, 1, 1, 1)).sum(0)
        return self.attn_pool(comb, attention_mask)          # (context_vector, attention_weights)

class VFHead(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x)

class VFModel(nn.Module):
    def __init__(self, name, unfreeze=2):
        super().__init__()
        self.encoder = GeneEncoder(name, unfreeze); self.head = VFHead(self.encoder.hidden_dim)
    def forward(self, ids, mask):
        ctx, attn = self.encoder(ids, mask)
        return self.head(ctx).squeeze(-1), ctx, attn

def contrastive_loss(emb, labels, temperature=0.1):
    emb = F.normalize(emb.float(), dim=1); labels = labels.view(-1)
    sim = emb @ emb.T / temperature
    pos = (labels[:, None] == labels[None, :]).float(); pos.fill_diagonal_(0)
    sim = sim - sim.max(dim=1, keepdim=True)[0].detach()
    e = torch.exp(sim) * (1 - torch.eye(sim.size(0), device=sim.device))
    logp = sim - torch.log(e.sum(1, keepdim=True) + 1e-8)
    return (-(pos * logp).sum(1) / pos.sum(1).clamp(min=1)).mean()

class SeqDataset(Dataset):
    def __init__(self, seqs, labels):
        self.s, self.y = list(seqs), np.asarray(labels, dtype=np.float32)
    def __len__(self):
        return len(self.s)
    def __getitem__(self, i):
        return self.s[i], self.y[i]

def collate(batch):
    seqs, ys = zip(*batch)
    enc = tokenizer(list(seqs), truncation=True, padding=True, max_length=CFG["max_len"], return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"], torch.tensor(np.asarray(ys), dtype=torch.float32)

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.no_grad()
def _eval_loader(model, dl, bce):
    model.eval(); probs, ys, tot, n = [], [], 0.0, 0
    for ids, mask, yb in dl:
        ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logit, ctx, _ = model(ids, mask)
        l = bce(logit.float(), yb) + CFG["beta"] * contrastive_loss(ctx, yb, CFG["tau"])
        tot += l.item() * len(yb); n += len(yb)
        probs.append(torch.sigmoid(logit.float()).cpu().numpy()); ys.append(yb.cpu().numpy())
    p, yy = np.concatenate(probs), np.concatenate(ys).astype(int)
    return tot / n, roc_auc_score(yy, p), accuracy_score(yy, p >= 0.5), f1_score(yy, p >= 0.5)

def train_stage1(seed, tr_idx, va_idx, tag):
    """Fine-tune on TRAIN, select the epoch on VAL AUC (the test fold is never touched). Resumable per epoch."""
    ckd = OUT / "ckpt"
    best_p, last_p, hist_p = ckd / f"{tag}_s{seed}_best.pt", ckd / f"{tag}_s{seed}_last.pt", ckd / f"{tag}_s{seed}_hist.json"
    set_seed(seed)
    model = VFModel(CFG["model_name"], CFG["unfreeze"]).to(DEVICE)
    if best_p.exists() and hist_p.exists() and json.load(open(hist_p)).get("done"):
        model.load_state_dict(torch.load(best_p, map_location=DEVICE)); model.eval()
        return model, json.load(open(hist_p))["history"]
    seqs, yv = df.seq.values, y_all
    g = torch.Generator(); g.manual_seed(seed)
    tr_dl = DataLoader(SeqDataset(seqs[tr_idx], yv[tr_idx]), batch_size=CFG["batch_size"], shuffle=True, generator=g, collate_fn=collate)
    va_dl = DataLoader(SeqDataset(seqs[va_idx], yv[va_idx]), batch_size=32, shuffle=False, collate_fn=collate)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"])
    scaler = make_scaler()
    n_pos = int(yv[tr_idx].sum()); n_neg = len(tr_idx) - n_pos
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE))
    start, best_auc, history = 0, -1.0, []
    if last_p.exists():
        ck = torch.load(last_p, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best_auc, history = ck["epoch"] + 1, ck["best_auc"], ck["history"]
        print(f"  resumed {tag} seed {seed} at epoch {start + 1}")
    for epoch in range(start, CFG["epochs"]):
        model.train(); t0 = time.time(); tot, nb, correct = 0.0, 0, 0
        for ids, mask, yb in tr_dl:
            ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logit, ctx, _ = model(ids, mask)
            loss = bce(logit.float(), yb) + CFG["beta"] * contrastive_loss(ctx, yb, CFG["tau"])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(yb); nb += len(yb); correct += ((logit.detach() > 0).float() == yb).sum().item()
        v_loss, v_auc, v_acc, v_f1 = _eval_loader(model, va_dl, bce)
        history.append(dict(epoch=epoch + 1, train_loss=tot / nb, train_acc=correct / nb, val_loss=v_loss, val_auc=v_auc, val_acc=v_acc, val_f1=v_f1))
        print(f"  [{tag} s{seed}] epoch {epoch+1}/{CFG['epochs']} train {tot/nb:.4f} | val loss {v_loss:.4f} acc {v_acc:.3f} f1 {v_f1:.3f} AUC {v_auc:.4f} | {time.time()-t0:.0f}s")
        if v_auc > best_auc:
            best_auc = v_auc; torch.save(model.state_dict(), best_p)
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), epoch=epoch, best_auc=best_auc, history=history), last_p)
    json.dump(dict(done=True, history=history), open(hist_p, "w"))
    model.load_state_dict(torch.load(best_p, map_location=DEVICE)); model.eval()
    return model, history

@torch.no_grad()
def infer(model, seqs, batch_size=32):
    """Pooled 480-d embedding + head probability for every sequence (sorted by length for speed, original order returned)."""
    model.eval(); seqs = list(seqs); order = np.argsort([len(s) for s in seqs])
    emb = np.zeros((len(seqs), model.encoder.hidden_dim), np.float32); prob = np.zeros(len(seqs), np.float32)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        enc = tokenizer([seqs[j] for j in idx], truncation=True, padding=True, max_length=CFG["max_len"], return_tensors="pt")
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logit, ctx, _ = model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
        emb[idx] = ctx.float().cpu().numpy(); prob[idx] = torch.sigmoid(logit.float()).cpu().numpy()
    return emb, prob

def get_layer_weights(model):
    return torch.softmax(model.encoder.layer_weight_logits.detach().cpu(), 0).numpy()

@torch.no_grad()
def frozen_mean_embed(seqs, batch_size=32):
    m = AutoModel.from_pretrained(CFG["model_name"]).to(DEVICE).eval()
    seqs = list(seqs); order = np.argsort([len(s) for s in seqs]); out = np.zeros((len(seqs), m.config.hidden_size), np.float32)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        enc = tokenizer([seqs[j] for j in idx], truncation=True, padding=True, max_length=CFG["max_len"], return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            h = m(**enc).last_hidden_state.float()
        mk = enc["attention_mask"].unsqueeze(-1).float()
        out[idx] = ((h * mk).sum(1) / mk.sum(1)).cpu().numpy()
    del m; free_gpu()
    return out

In [ ]:
# §10  (items 3, 11, 12)  Frozen-ESM baseline, then: for every regime x seed -> train Stage 1, embed, fit meta-models, score the genome
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

if CFG["frozen_baseline"]:
    cp = OUT / "cache" / "frozen_mean_emb.npy"
    E = np.load(cp) if cp.exists() else frozen_mean_embed(df.seq.tolist())
    if not cp.exists():
        np.save(cp, E)
    lr_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1)).fit(E[tr_m], y_all[tr_m])
    BASELINES["frozen_esm2_meanpool+LR"] = eval_probs(yt, lr_clf.predict_proba(E[te_m])[:, 1])
    print("frozen ESM-2 + LR (cluster-split test):", {k: round(v, 4) for k, v in BASELINES["frozen_esm2_meanpool+LR"].items()})

REGIMES = {"cluster": cluster_split}
if CFG["retrain_random"]:
    REGIMES["random"] = random3

def feature_sets(emb, cdd, bits):
    fs = {"emb_only": emb, "m2": np.c_[emb, cdd]}
    for v, b in bits.items():
        fs[f"m3_{v}"] = np.c_[emb, cdd, b]
    return fs

def make_meta(seed):
    return xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric="logloss", random_state=seed, n_jobs=-1)

RUNS = {}
genome_seqs = genes.seq.tolist()
g_bits_by_variant = {v: g_bits for v in BITS}            # genome genes: raw best bitscore, whatever variant the model was trained with
for regime, split in REGIMES.items():
    tr_idx, va_idx, te_idx = [np.where(split == s)[0] for s in ("train", "val", "test")]
    for seed in CFG["seeds"]:
        key, cache_p = (regime, seed), OUT / "cache" / f"run_{regime}_seed{seed}.pkl"
        if cache_p.exists():
            RUNS[key] = pickle.load(open(cache_p, "rb")); print("loaded cached run", key); continue
        t0 = time.time()
        model, hist = train_stage1(seed, tr_idx, va_idx, regime)
        emb, p_head = infer(model, df.seq.tolist())
        run = dict(history=hist, probs={"head": p_head}, genome={}, layer_weights=get_layer_weights(model))
        fsets = feature_sets(emb, cdd_train, BITS)
        if regime == "cluster":
            g_emb, g_head = infer(model, genome_seqs)
            run["genome"]["head"] = g_head
            gsets = feature_sets(g_emb, g_cdd, g_bits_by_variant)
        for name, X in fsets.items():
            clf = make_meta(seed).fit(X[tr_idx], y_all[tr_idx])
            run["probs"][name] = clf.predict_proba(X)[:, 1]
            if regime == "cluster":
                run["genome"][name] = clf.predict_proba(gsets[name])[:, 1]
        pickle.dump(run, open(cache_p, "wb")); RUNS[key] = run
        del model; free_gpu()
        te_auc = {k: round(roc_auc_score(y_all[te_idx], v[te_idx]), 4) for k, v in run["probs"].items()}
        print(f"== {regime} seed {seed} done in {(time.time()-t0)/60:.1f} min | test AUC {te_auc}")

In [ ]:
# §11  (items 5, 11, 12)  Results tables: mean ± SD over seeds, baselines, cluster-bootstrap CIs, train->genome feature shift
seeds = CFG["seeds"]
rows = []
for regime, sp in REGIMES.items():
    te = np.where(sp == "test")[0]
    for nm in RUNS[(regime, seeds[0])]["probs"]:
        ms = [eval_probs(y_all[te], RUNS[(regime, s)]["probs"][nm][te]) for s in seeds]
        r = dict(regime=regime, model=nm, n_seeds=len(seeds), auc_mean=float(np.mean([m["auc"] for m in ms])))
        for k in ["auc", "ap", "acc", "f1", "brier"]:
            r[k] = fmt_ms([m[k] for m in ms])
        rows.append(r)
main_tbl = pd.DataFrame(rows)
bl_rows = [dict(regime="cluster", model=f"[baseline] {k}", n_seeds=1, auc_mean=v["auc"], auc=f"{v['auc']:.4f}", ap=f"{v['ap']:.4f}",
                acc=f"{v['acc']:.4f}" if "acc" in v else "-", f1=f"{v['f1']:.4f}" if "f1" in v else "-",
                brier=f"{v['brier']:.4f}" if "brier" in v else "-") for k, v in BASELINES.items()]
main_tbl = pd.concat([main_tbl, pd.DataFrame(bl_rows)], ignore_index=True)
save_table(main_tbl.set_index(["regime", "model"]), "main_results")
print("=== TEST-SET RESULTS (mean ± SD over seeds; baselines use one run) ===")
print(main_tbl.drop(columns="auc_mean").to_string(index=False))
print("\nFirst-draft numbers (random split, leaky): head 0.957 | M2 0.9672 | M3 0.9744 -> compare with the 'cluster' rows above")

# ---- paired, cluster-aware bootstrap on the primary regime (seed 0) ----
te = np.where(cluster_split == "test")[0]; s0 = seeds[0]
P = RUNS[("cluster", s0)]["probs"]
pairs = [("m3_self", "m2"), ("m2", "emb_only"), ("emb_only", "head"), ("m3_family", "m3_self"), ("m3_cap90", "m3_self")]
brows = []
for a, b in pairs:
    r = paired_bootstrap_delta(y_all[te], P[a][te], P[b][te], n=CFG["boot"], groups=groups[te])
    per_seed = [roc_auc_score(y_all[te], RUNS[("cluster", s)]["probs"][a][te]) - roc_auc_score(y_all[te], RUNS[("cluster", s)]["probs"][b][te]) for s in seeds]
    brows.append(dict(comparison=f"{a} - {b}", delta_auc=round(r["delta"], 4), ci95=f"[{r['lo']:.4f}, {r['hi']:.4f}]",
                      p_delta_le_0=round(r["p_le0"], 3), across_seed_delta=fmt_ms(per_seed),
                      significant="YES" if (r["lo"] > 0 or r["hi"] < 0) else "no"))
boot_tbl = save_table(pd.DataFrame(brows).set_index("comparison"), "bootstrap_deltas")
print("\n=== paired cluster-bootstrap of AUC differences (", CFG["boot"], "resamples ) ===")
print(boot_tbl.to_string())
ci_rows = []
for nm in P:
    a, lo, hi = boot_auc_ci(y_all[te], P[nm][te], n=CFG["boot"], groups=groups[te])
    ci_rows.append(dict(model=nm, auc=round(a, 4), ci95=f"[{lo:.4f}, {hi:.4f}]"))
print("\n=== single-model AUC with 95% cluster-bootstrap CI (seed", s0, ") ===")
print(save_table(pd.DataFrame(ci_rows).set_index("model"), "auc_ci").to_string())

if "random" in REGIMES:
    cmp = main_tbl[main_tbl.regime.isin(["cluster", "random"]) & ~main_tbl.model.str.startswith("[baseline]")].pivot(index="model", columns="regime", values="auc_mean")
    cmp["drop_when_homology_controlled"] = cmp["random"] - cmp["cluster"]
    print("\n=== random vs homology-aware split (same recipe, same seeds) ===")
    print(save_table(cmp.round(4), "random_vs_cluster").to_string())

# ---- item 5: is the BLAST feature distributed like the genome's? ----
b_tr, b_ge = BITS["self"], g_bits
shift = pd.DataFrame({
    "frac_with_hit": [np.mean(b_tr[y_all == 1] > 0), np.mean(b_tr[y_all == 0] > 0), np.mean(b_ge > 0)],
    "median_bitscore_if_hit": [np.median(b_tr[(y_all == 1) & (b_tr > 0)]), np.median(b_tr[(y_all == 0) & (b_tr > 0)]), np.median(b_ge[b_ge > 0]) if (b_ge > 0).any() else np.nan],
    "frac_bitscore>=300": [np.mean(b_tr[y_all == 1] >= 300), np.mean(b_tr[y_all == 0] >= 300), np.mean(b_ge >= 300)],
    "frac_with_CDD_domain": [np.mean(cdd_train[y_all == 1] > 0), np.mean(cdd_train[y_all == 0] > 0), np.mean(g_cdd > 0)],
    "median_CDD_count_if_any": [np.median(cdd_train[(y_all == 1) & (cdd_train > 0)]), np.median(cdd_train[(y_all == 0) & (cdd_train > 0)]), np.median(g_cdd[g_cdd > 0]) if (g_cdd > 0).any() else np.nan],
}, index=["train positives", "train negatives", "genome genes"]).round(3)
print("\n=== evidence-feature distribution: training vs genome (same definitions) ===")
print(save_table(shift, "feature_shift_train_vs_genome").to_string())

In [ ]:
# §12  (item 7)  Calibration, thresholds and prevalence-adjusted expected false positives (primary model, ensemble over seeds)
PRIMARY = CFG["primary_model"]
va = np.where(cluster_split == "val")[0]; te = np.where(cluster_split == "test")[0]
def ens(name, idx):
    return np.mean([RUNS[("cluster", s)]["probs"][name][idx] for s in seeds], axis=0)
pv, pt, yv, ytt = ens(PRIMARY, va), ens(PRIMARY, te), y_all[va], y_all[te]
def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
iso = IsotonicRegression(out_of_bounds="clip").fit(pv, yv)                 # fitted on VAL only
platt = LogisticRegression(C=1e6).fit(_logit(pv)[:, None], yv)
p_iso = iso.predict(pt); p_platt = platt.predict_proba(_logit(pt)[:, None])[:, 1]
cal_tbl = pd.DataFrame({
    "Brier": [brier_score_loss(ytt, pt), brier_score_loss(ytt, p_iso), brier_score_loss(ytt, p_platt)],
    "ECE":   [ece_score(ytt, pt), ece_score(ytt, p_iso), ece_score(ytt, p_platt)],
    "AUC":   [roc_auc_score(ytt, pt), roc_auc_score(ytt, p_iso), roc_auc_score(ytt, p_platt)]},
    index=["raw", "isotonic", "platt"]).round(4)
print(f"=== calibration of {PRIMARY} on the cluster-split TEST fold (calibrators fitted on VAL) ===")
print(save_table(cal_tbl, "calibration").to_string())

def at_threshold(y, p, t):
    pred = p >= t; tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    return dict(threshold=t, TP=tp, FP=fp, FN=fn, TN=tn, precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1), FPR=fp / max(fp + tn, 1))
thr_tbl = pd.DataFrame([at_threshold(ytt, pt, t) for t in [0.5, 0.8, 0.9, 0.95]]).set_index("threshold")
print("\n=== precision / recall / FPR at fixed thresholds (test fold, balanced-prevalence data) ===")
print(save_table(thr_tbl.round(4), "threshold_table").to_string())

# label-shift correction: precision at a realistic genome prevalence pi, using the test TPR/FPR at each threshold
N_G = len(genes); rows = []
for t, r in thr_tbl.iterrows():
    for pi in CFG["prevalences"]:
        tpr, fpr = r["recall"], r["FPR"]
        rows.append(dict(threshold=t, assumed_prevalence=pi, precision_at_prevalence=tpr * pi / max(tpr * pi + fpr * (1 - pi), 1e-12),
                         expected_TP_in_genome=N_G * pi * tpr, expected_FP_in_genome=N_G * (1 - pi) * fpr,
                         expected_calls_in_genome=N_G * (pi * tpr + (1 - pi) * fpr)))
exp_tbl = save_table(pd.DataFrame(rows).round(3).set_index(["threshold", "assumed_prevalence"]), "prevalence_adjusted_expectations")
print(f"\n=== what to expect in a genome with {N_G} genes if the true VF prevalence is pi (NOTE: FPR measured on dataset negatives,"
      " which are probably easier than a genome's housekeeping genes -> these are optimistic) ===")
print(exp_tbl.to_string())

In [ ]:
# §13  (item 14)  All genome analyses on the Prodigal set with TRUE coordinates (ensemble over seeds, cluster-regime models)
GPROB = {nm: np.mean([RUNS[("cluster", s)]["genome"][nm] for s in seeds], axis=0) for nm in RUNS[("cluster", seeds[0])]["genome"]}
res = genes.drop(columns=["seq"]).copy()
res["cdd_count"] = g_cdd.astype(int); res["blast_bitscore"] = g_bits
res["has_blast"] = res.gene_id.isin(set(gblast.query_id))
bh = gblast.sort_values(["evalue", "bitscore"], ascending=[True, False]).drop_duplicates("query_id").set_index("query_id")
for c in ["subject_id", "subject_title", "pident", "length", "evalue"]:
    res[f"blast_{c}"] = res.gene_id.map(bh[c]) if c in bh.columns else np.nan
for nm, p in GPROB.items():
    res[f"p_{nm}"] = p
res["p_primary"] = res[f"p_{PRIMARY}"]; res["p_primary_iso"] = iso.predict(res.p_primary.values)
res["blast_cov"] = (res.blast_length / res.aa_len).clip(upper=1.0)
THR = CFG["hc_threshold"]; res["high_conf"] = res.p_primary > THR
res["tier"] = "no_evidence"
res.loc[res.has_blast | (res.cdd_count > 0), "tier"] = "corroborated"
res.loc[res.has_blast & (res.cdd_count > 0), "tier"] = "strongly_corroborated"
res.to_csv(OUT / "tables" / "genome_predictions_prodigal_v2.csv", index=False)

print(f"=== genome: {len(res)} genes | score = {PRIMARY} (ensemble of {len(seeds)} seeds) ===")
t5 = pd.crosstab(res.tier, res.high_conf).rename(columns={False: f"p<={THR}", True: f"p>{THR}"})
t5["all"] = t5.sum(1); t5.loc["TOTAL"] = t5.sum()
print("\nTable 5 (tiers, filled):"); print(save_table(t5, "table5_tiers").to_string())
print(f"genes above {THR}: raw {int(res.high_conf.sum())} ({res.high_conf.mean():.1%}) | isotonic-calibrated >{THR}: {int((res.p_primary_iso > THR).sum())}")
cnt_obs = {t: int((res.p_primary >= t).sum()) for t in [0.5, 0.8, 0.9, 0.95]}
print("observed genome calls per threshold:", cnt_obs, "-> compare with the expected counts in §12 to judge the plausible false-positive load")
print("scores of every model above", THR, ":", {nm: int((p > THR).sum()) for nm, p in GPROB.items()})

# ---- evidence concordance (UpSet counts) ----
ups = res.assign(HighConf=res.high_conf, BLAST=res.has_blast, CDD=res.cdd_count > 0).groupby(["HighConf", "BLAST", "CDD"]).size().rename("n").reset_index().sort_values("n", ascending=False)
print("\nevidence concordance (UpSet-style counts):"); print(save_table(ups.set_index(["HighConf", "BLAST", "CDD"]), "upset_counts").to_string())

print("\n=== top-15 predictions ==="); 
print(res.sort_values("p_primary", ascending=False).head(15)[["gene_id", "contig", "start", "end", "aa_len", "p_primary", "p_primary_iso", "has_blast", "cdd_count", "tier"]].round(4).to_string(index=False))

# ---- candidate loci with real coordinates ----
def cluster_loci(g, gap):
    g = g.sort_values(["contig", "start"]).reset_index(drop=True); ids, c, pc, pe = [], 0, None, None
    for contig, s, e in zip(g.contig, g.start, g.end):
        if contig != pc or s - pe > gap:
            c += 1; pe = e
        else:
            pe = max(pe, e)
        ids.append(c); pc = contig
    g["locus"] = ids
    return g
loc_rows = []
for nm, mask in [("high-confidence, no evidence", res.high_conf & (res.tier == "no_evidence")), ("high-confidence, all", res.high_conf)]:
    L = cluster_loci(res[mask], CFG["locus_gap_bp"]); sizes = L.groupby("locus").size()
    loc_rows.append(dict(set=nm, genes=len(L), loci=int(sizes.size), singletons=int((sizes == 1).sum()), loci_ge6_genes=int((sizes >= 6).sum()), largest=int(sizes.max()) if len(sizes) else 0))
    if nm.endswith("no evidence"):
        L.to_csv(OUT / "tables" / "loci_no_evidence.csv", index=False)
print(f"\n=== candidate loci (genes <= {CFG['locus_gap_bp']} bp apart on the same contig) ==="); print(save_table(pd.DataFrame(loc_rows).set_index("set"), "loci").to_string())

# ---- how do BLAST-confirmed genes score? (old section 4.5) ----
bc = res[res.has_blast]
print(f"\n=== BLAST-confirmed genes (n={len(bc)}) ===")
print(f"median p {bc.p_primary.median():.3f} | mean {bc.p_primary.mean():.3f} | <0.8: {(bc.p_primary<0.8).mean():.1%} | <0.5: {(bc.p_primary<0.5).mean():.1%}")
cors = {c: spearmanr(bc.p_primary, bc[c], nan_policy="omit")[0] for c in ["aa_len", "blast_pident", "blast_bitscore", "blast_cov"]}
print("Spearman(p, x):", {k: round(float(v), 3) for k, v in cors.items()})
strong = bc[bc.blast_pident > 40]
print(f"hits with identity>40%: {len(strong)} | scoring <0.1: {int((strong.p_primary<0.1).sum())} | scoring >0.9: {int((strong.p_primary>0.9).sum())}")

# ---- housekeeping analysis (old 4.6), best-overall-hit fix ----
HK_VFG = ["VFG046465", "VFG013248"]
res["best_vfg"] = res.blast_subject_id.astype(str).str.extract(r"^(VFG\d+)")[0]
hk = res[res.best_vfg.isin(HK_VFG)]
print(f"\n=== genes whose BEST VFDB hit is {HK_VFG}: n={len(hk)} | median p {hk.p_primary.median():.3f} | >0.8: {int((hk.p_primary>0.8).sum())} | 0.5-0.8: {int(((hk.p_primary>0.5)&(hk.p_primary<=0.8)).sum())}")
t7 = hk[hk.p_primary > 0.5].sort_values("p_primary", ascending=False)[["gene_id", "contig", "start", "end", "p_primary", "blast_pident", "blast_length", "aa_len", "blast_cov"]].round(3)
print("Table 7 (p>0.5):"); print(save_table(t7.set_index("gene_id"), "table7_housekeeping_hits").to_string())
res["hk_family"] = res.blast_subject_title.map(classify_family)
if res.hk_family.notna().any():
    fam_rows = []
    for fam_name, g in res[res.hk_family.notna()].groupby("hk_family"):
        fam_rows.append(dict(family=fam_name, genes=len(g), median_p=round(g.p_primary.median(), 4), n_gt_0_8=int((g.p_primary > 0.8).sum()),
                             n_gt_0_8_and_cov_gt_80=int(((g.p_primary > 0.8) & (g.blast_cov > 0.8)).sum())))
    print("\nTable 8 (family = keyword in the title of each gene's BEST overall hit):")
    print(save_table(pd.DataFrame(fam_rows).set_index("family").sort_values("genes", ascending=False), "table8_housekeeping_families").to_string())
    ev_hc = res[res.high_conf & (res.tier != "no_evidence")]
    print(f"evidence-backed high-confidence genes: {len(ev_hc)} | housekeeping-family keyword in best hit: {int(ev_hc.hk_family.notna().sum())} ({ev_hc.hk_family.notna().mean():.1%})")
print(f"\nNOTE: the {int(((res.tier=='no_evidence') & res.high_conf).sum())} high-confidence NO-EVIDENCE genes have no hit title -> annotate them (e.g. eggNOG-mapper) to count housekeeping genes among them.")

In [ ]:
# §14  Figures + export
fig_dir = OUT / "figures"
P0 = RUNS[("cluster", seeds[0])]["probs"]
# ROC + PR (test fold, seed 0)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for nm in ["head", "emb_only", "m2", "m3_self", "m3_family"]:
    fpr, tpr, _ = roc_curve(ytt, P0[nm][te]); ax[0].plot(fpr, tpr, label=f"{nm} (AUC {roc_auc_score(ytt, P0[nm][te]):.3f})")
    pr, rc, _ = precision_recall_curve(ytt, P0[nm][te]); ax[1].plot(rc, pr, label=nm)
ax[0].plot([0, 1], [0, 1], "k--", alpha=.4); ax[0].set(xlabel="FPR", ylabel="TPR", title="ROC - cluster-split test"); ax[0].legend()
ax[1].set(xlabel="recall", ylabel="precision", title="Precision-recall - cluster-split test"); ax[1].legend()
plt.tight_layout(); plt.savefig(fig_dir / "roc_pr.png", dpi=150); plt.show()

# reliability
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for lab, p in [("raw", pt), ("isotonic", p_iso)]:
    bins = np.linspace(0, 1, 11); ix = np.clip(np.digitize(p, bins[1:-1]), 0, 9)
    xs = [p[ix == b].mean() for b in range(10) if (ix == b).sum() > 5]; ys = [ytt[ix == b].mean() for b in range(10) if (ix == b).sum() > 5]
    ax.plot(xs, ys, "o-", label=lab)
ax.plot([0, 1], [0, 1], "k--", alpha=.4); ax.set(xlabel="mean predicted probability", ylabel="fraction of VFs", title=f"Reliability ({PRIMARY}, test)"); ax.legend()
plt.tight_layout(); plt.savefig(fig_dir / "reliability.png", dpi=150); plt.show()

# genome score distribution by tier + layer weights + learning curves
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for t, g in res.groupby("tier"):
    ax[0].hist(g.p_primary, bins=40, alpha=.5, label=f"{t} (n={len(g)})")
ax[0].axvline(THR, color="r", ls="--"); ax[0].set(xlabel=f"{PRIMARY} score", ylabel="genes", title="Genome (Prodigal) score by evidence tier"); ax[0].legend()
lw = np.mean([RUNS[("cluster", s)]["layer_weights"] for s in seeds], axis=0)
ax[1].bar(range(len(lw)), lw); ax[1].axhline(1 / len(lw), color="k", ls="--", alpha=.5); ax[1].set(xlabel="layer (0 = embeddings)", ylabel="softmax weight", title="Learned layer weights (mean over seeds)")
for s in seeds:
    h = pd.DataFrame(RUNS[("cluster", s)]["history"]); ax[2].plot(h.epoch, h.val_auc, "o-", label=f"seed {s}")
ax[2].set(xlabel="epoch", ylabel="val AUC", title="Stage-1 validation AUC (cluster split)"); ax[2].legend()
plt.tight_layout(); plt.savefig(fig_dir / "genome_layers_curves.png", dpi=150); plt.show()

try:
    from upsetplot import UpSet, from_indicators
    ind = pd.DataFrame({"High confidence": res.high_conf, "BLAST hit": res.has_blast, "CDD domain": res.cdd_count > 0})
    ind = ind[ind.any(axis=1)]
    UpSet(from_indicators(ind.columns.tolist(), ind), subset_size="count", show_counts=True).plot()
    plt.savefig(fig_dir / "upset.png", dpi=150, bbox_inches="tight"); plt.show()
except Exception as e:
    print("upset plot skipped:", e)

# export a compact summary + the exact configuration
summary = dict(config=CFG, n_sequences=len(df), n_genes=len(genes), split_sizes={s: int((cluster_split == s).sum()) for s in ["train", "val", "test"]},
               leakage_old_random_split=leak_tbl.loc["old random 80/20"].to_dict(), primary_model=PRIMARY,
               genes_above_threshold=int(res.high_conf.sum()), versions=dict(torch=torch.__version__, xgboost=xgb.__version__, numpy=np.__version__, pandas=pd.__version__))
json.dump(summary, open(OUT / "tables" / "run_summary.json", "w"), indent=2, default=str)
print("\nAll tables ->", OUT / "tables"); print("All figures ->", fig_dir)
print(sorted(p.name for p in (OUT / "tables").glob("*.csv")))

### What to paste back into the paper after a full run
* **Methods 3.1 / 3.3** → `split_leakage.csv` (how much the old split leaked), `dataset_audit.csv`, `shortcut_baselines.csv`; epochs are now chosen on a separate *val* fold.
* **Methods 3.4** → the CDD rule is now *domain bit-score ≥ `dom_score_min`* (database-size independent); BLAST feature = best bitscore with the de-leaking variant stated (`blast_feature_variants.csv`).
* **Results 4.1–4.3** → `main_results.csv` (mean ± SD, baselines), `bootstrap_deltas.csv`, `auc_ci.csv`, `random_vs_cluster.csv` (if `retrain_random=True`), `feature_shift_train_vs_genome.csv`.
* **Results 4.4–4.8** → `table5_tiers.csv`, `upset_counts.csv`, `loci.csv`, `table7_housekeeping_hits.csv`, `table8_housekeeping_families.csv`, `genome_predictions_prodigal_v2.csv`.
* **Discussion 5.3** → `calibration.csv`, `threshold_table.csv`, `prevalence_adjusted_expectations.csv`.